In [ ]:
!pip3 install flask

In [2]:
import pandas as pd
import requests
import json
import time
import csv
import io
import Levenshtein
from urllib.parse import urlparse
from flask import Flask, jsonify
from requests.auth import HTTPBasicAuth

/Users/sooreoluwa/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [49]:

API_KEY = "0608ffe9-6566-4c1f-831e-0934c7e18547"
BASE_URL = "https://pro-api.coinmarketcap.com"

headers = {
    "Accepts": "application/json",
    "X-CMC_PRO_API_KEY": API_KEY,
}

def fetch_cryptocurrencies():
    page = 1
    all_crypto = []
    max_pages = 20  # Safety limit for free tier
    limit = 1000  # Max allowed per call

    while page <= max_pages:
        parameters = {
            "start": (page - 1) * limit + 1,
            "limit": limit,
            "convert": "USD"
        }

        try:
            response = requests.get(f"{BASE_URL}/v1/cryptocurrency/listings/latest", headers=headers, params=parameters)
            
            if response.status_code == 200:
                data = response.json()
                crypto_data = data.get("data", [])
                
                if not crypto_data:
                    break
                
                # Get all coin IDs from current page
                coin_ids = [str(coin['id']) for coin in crypto_data]
                
                # Split into chunks of 200 (max allowed per metadata call)
                chunks = [coin_ids[i:i+200] for i in range(0, len(coin_ids), 200)]
                metadata = {}

                # Fetch metadata for each chunk
                for chunk in chunks:
                    params_info = {
                        'id': ','.join(chunk),
                        'aux': 'urls'
                    }
                    try:
                        response_info = requests.get(f"{BASE_URL}/v2/cryptocurrency/info", 
                                                   headers=headers, 
                                                   params=params_info)
                        
                        if response_info.status_code == 200:
                            info_data = response_info.json()
                            metadata.update(info_data.get('data', {}))
                            time.sleep(1)  # Additional delay for metadata calls
                        elif response_info.status_code == 429:
                            print("Metadata rate limit hit. Waiting 60 seconds...")
                            time.sleep(60)
                        else:
                            print(f"Metadata error {response_info.status_code}: {response_info.text}")
                    except requests.exceptions.RequestException as e:
                        print(f"Metadata request failed: {e}")

                # Add website URLs to crypto data
                for coin in crypto_data:
                    coin_id = str(coin['id'])
                    urls = metadata.get(coin_id, {}).get('urls', {})
                    websites = urls.get('website', [])
                    coin['url'] = websites[0] if websites else ""

                all_crypto.extend(crypto_data)
                print(f"Fetched page {page} with {len(crypto_data)} currencies")
                page += 1
                time.sleep(1)  # Original rate limit delay

            elif response.status_code == 429:
                print("Rate limit hit. Waiting 60 seconds...")
                time.sleep(60)

            else:
                print(f"Error {response.status_code}: {response.text}")
                break

        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            break

    return all_crypto

In [64]:
# function to fetch exchanges with pagination
def fetch_exchanges(url):
    page = 1
    all_exchanges = []
    
    while True:
        # Add the page parameter to the request
        params = {"per_page": 250, "page": page} 
        try:
            # Make the request
            response = requests.get(url, params=params)
            # Check if the request was successful
            if response.status_code == 200:
                data = response.json()
                # If no data is returned, stop the loop
                if not data:
                    break
                # Add the fetched data to the list
                all_exchanges.extend(data)
                print(f"Fetched page {page} with {len(data)} exchanges.")
                # Move to the next page
                page += 1
                # Add a delay to avoid hitting the rate limit
                time.sleep(1) 
    
            elif response.status_code == 429:
                # Handle rate limit error
                print("Rate limit exceeded. Waiting for 10 seconds before retrying...")
                time.sleep(60)  # Wait 60 seconds before retrying
            
            else:
                # Handle other errors
                print(f"Failed to fetch data. Status code: {response.status_code}. Reason: {response.reason}")
                break
        
        except requests.exceptions.RequestException as e:
            # Handle network errors
            print(f"Network error: {e}")
            break
    
    return all_exchanges

# Function to fetch exchanges from DeFi Llama
def fetch_defi(url):
    # Make the request
    response = requests.get(url)
    if response.status_code == 200:
        # Return the JSON data
        return response.json() 
    else:
        print(f"Failed to fetch data. Status code: {response.status_code}. Reason: {response.reason}")
        
        return []

# function to fetch the blacklist url
def blacklist_map(items):
    # Create a list to store the name-url mappings
    blacklist_map = []

    # Process each URL in the blacklist
    for url in items:
        # Extract the text before the domain extension
        name = url.split('.')[0]
        # Create a dictionary for the name and URL
        blacklist_map.append({'name': name, 'url': url})

    return blacklist_map

def format_domain(domain):
    if not domain.startswith(('https://', 'http://', 'www.')):
        return 'https://' + domain
    return domain


In [33]:
# fetch centralized exchanges from coingecko
cex_url = "https://api.coingecko.com/api/v3/exchanges"
gecko_data = fetch_exchanges(cex_url)

# Print the total number of exchanges fetched
print(f"Total exchanges fetched: {len(gecko_data)}")

Fetched page 1 with 250 exchanges.
Fetched page 2 with 250 exchanges.
Fetched page 3 with 250 exchanges.
Fetched page 4 with 194 exchanges.
Total exchanges fetched: 944


In [ ]:
# fetch every exchange protocol from defi llama
url = "https://api.llama.fi/protocols"
llama_data = fetch_defi(url)

# Print the total number of exchanges fetched
print(f"Total exchanges fetched: {len(llama_data)}")

Total exchanges fetched: 5726


In [50]:
coinmarketcap_data = fetch_cryptocurrencies()
coinmarketcap_df = pd.DataFrame(coinmarketcap_data)
coinmarketcap_df = coinmarketcap_df[["name", "url"]]

# Print the total number of exchanges fetched
print(f"Total exchanges fetched: {len(coinmarketcap_df)}")

Fetched page 1 with 1000 currencies
Fetched page 2 with 1000 currencies
Fetched page 3 with 1000 currencies
Fetched page 4 with 1000 currencies
Fetched page 5 with 1000 currencies
Fetched page 6 with 1000 currencies
Fetched page 7 with 1000 currencies
Fetched page 8 with 1000 currencies
Fetched page 9 with 1000 currencies
Fetched page 10 with 1000 currencies
Fetched page 11 with 391 currencies
Total exchanges fetched: 10391


In [58]:
# remove row with None value
coinmarketcap_df = coinmarketcap_df[coinmarketcap_df["url"] != ""].drop_duplicates(subset=["url"], keep="first").reset_index(drop=True)
coinmarketcap_df

,url
0,https://bitcoin.org/
1,https://www.ethereum.org/
2,https://tether.to
3,https://xrpl.org/
4,https://bnbchain.org/en
...,...
9945,https://odapp.io/
9946,https://hector.finance
9947,https://metapool.app/dapp/mainnet/meta/
9948,https://www.yieldnest.finance/


In [59]:
crypto_exchanges = llama_data + gecko_data

# Create a set to store unique URLs
unique_urls = set()

# Add URLs from cex_data to the set
for exchange in gecko_data:
    # Normalize by stripping trailing slashes
    unique_urls.add(exchange['url'].rstrip('/')) 

# Create a list to store unique exchanges
unique_exchanges = []

# Add exchanges from cex_data to the unique list
for exchange in gecko_data:
    if exchange['url'].rstrip('/') in unique_urls:
        unique_exchanges.append(exchange)

# Add exchanges from ex_data to the unique list
for exchange in llama_data:
    if exchange['url'].rstrip('/') in unique_urls:
        unique_exchanges.append(exchange)

# Print the total number of unique exchanges fetched
print(f"Total unique exchanges fetched: {len(unique_exchanges)}")

Total unique exchanges fetched: 1172


In [60]:
with open("legit-exchanges.json", "w") as file:
    json.dump(unique_exchanges, file)

In [ ]:
# # fetch other legitimate websites asides crypto
# # to ensure balance in datasets.
# other_websites = "https://raw.githubusercontent.com/seigdev/resources/refs/heads/main/top-websites.csv"

# other_raw = fetch_data_csv(other_websites)

# others_df = pd.DataFrame(other_raw)

# others_df = others_df.drop(columns=0)

# others_df = others_df.rename(columns={1: 'url'})

In [62]:

df_raw = pd.read_json("legit-exchanges.json")

# copy the name and urls columns to a new dataframe
crypto_df = df_raw[['url']].copy()

crypto_df["url"]   = crypto_df["url"].apply(format_domain)

coinmarketcap_df = coinmarketcap_df[["url"]].copy()

legit_df = pd.concat([crypto_df, coinmarketcap_df], ignore_index=True)

# assign string labels to legitimate urls
legit_df.loc[:, 'label'] = 'legit'

# assign string labels to legitimate urls
legit_df.loc[:, 'label_no'] = 0

# select first 100
legit_df = legit_df[:11000]

legit_df["url"]

0                                 https://www.binance.com/
1                                  https://www.bitget.com/
2                                    https://www.bybit.com
3                                https://www.coinbase.com/
4                              https://crypto.com/exchange
                               ...                        
10995                                   https://sherex.io/
10996                              https://mubarakcz.club/
10997                               https://amerotoken.com
10998                              https://shrekmoon.club/
10999    https://coinmarketcap.com/dexscan/bsc/0xd9dced...
Name: url, Length: 11000, dtype: object

In [65]:
# url to fetch scam urls from eth-phishing-detect
scam_url = "https://raw.githubusercontent.com/MetaMask/eth-phishing-detect/master/src/config.json"
scam_ex = fetch_defi(scam_url)

# fetch the list of blacklist urls
blacklist = scam_ex["blacklist"]

In [66]:
blacklist = blacklist_map(blacklist)

In [67]:
with open("scam-exchanges.json", "w") as file:
    json.dump(blacklist, file)

In [70]:
blacklist_raw = pd.read_json("scam-exchanges.json")

# copy the name and urls columns to a new dataframe
scam_df = blacklist_raw[['url']].copy()

scam_df["url"] = scam_df["url"].apply(format_domain)

# assign string labels to legitimate urls
scam_df.loc[:, 'label'] = 'scam'

# assign string labels to legitimate urls
scam_df.loc[:, 'label_no'] = 1

# select first 100
# scam_df = scam_df.sample(n=100000, random_state=42).reset_index(drop=True)
scam_df = scam_df[:50000]

scam_df

,url,label,label_no
0,https://ogntoken-migration.icu,scam,1
1,https://fazla-rabby-rady.github.io,scam,1
2,https://polymaraket.com,scam,1
3,https://predictdex.com,scam,1
4,https://polymarket.mx,scam,1
...,...,...,...
49995,https://kavalink.live,scam,1
49996,https://kavalink.net,scam,1
49997,https://kavalink.vip,scam,1
49998,https://kavalink.xyz,scam,1


In [71]:
# merge both the legit and scam urls together
urls_df = pd.concat([legit_df, scam_df], ignore_index=True)

# shuffle the urls across the dataframe
urls_df = urls_df.sample(frac=1, random_state=42).reset_index(drop=True)

urls_df

,url,label,label_no
0,https://combonetwork.io/,legit,0
1,https://vote-curve.net,scam,1
2,https://metamaskwalletextension32.mypixieset.com,scam,1
3,https://pyth.capital,scam,1
4,https://derive.xyz,legit,0
...,...,...,...
60995,https://xrp20coins.top,scam,1
60996,https://dojoethm.live,scam,1
60997,https://app.elk.finance/swap,legit,0
60998,https://metamaskwallet.gitbook.io,scam,1


In [72]:
urls_df.to_json('crypto_data.json', orient='records', lines=False)